# Graph vs search — локальные модели (LangChain)

Повтор eval из [graph-memory-starter](https://github.com/Glitch-Cat-Club/graph-memory-starter): один вопрос, два условия.

- **search** — LangChain-агент сам вызывает инструменты Grep/Read по `corpus-before/`.
- **graph** — SQLite-обход (`src/recall.py`) подставляет факты в промпт; модель **без тулов**.

Модель меняется в ячейке **Config**. По умолчанию — Ollama (`ChatOllama`). Нужны Python 3.10+, запущенный [Ollama](https://ollama.com) и модель с tool calling (например `qwen2.5:7b` или `llama3.1`).

```bash
ollama pull qwen2.5:7b
```

## 1. Зависимости

In [1]:
%pip install -qU "langchain>=1.0" langchain-ollama langchain-core

Note: you may need to restart the kernel to use updated packages.


## 2. Config — сюда подставляете модель

`PROVIDER = "ollama"` и `MODEL` — то, что меняете между прогонами. Для LM Studio / llama.cpp OpenAI-совместимого сервера поставьте `PROVIDER = "openai_compat"` и укажите `OPENAI_BASE` (нужен пакет `langchain-openai`).

In [25]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "recall.py").exists() else HERE.parent
if not (ROOT / "src" / "recall.py").exists():
    raise FileNotFoundError("Open this notebook from notebooks/ or the repo root (the directory that contains src/).")

PROVIDER = "openai_compat"  # "ollama" | "openai_compat"
MODEL = "gpt-oss-20b"  # ollama list / имя модели на локальном сервере
OLLAMA_URL = "http://192.168.0.55:1234"
OPENAI_BASE = "http://192.168.0.55:1234/v1"  # LM Studio, llama.cpp server, vLLM
OPENAI_API_KEY = "not-needed"

SEARCH_DIR = ROOT / "corpus-before"  # только unstructured docs
QUESTION = "A customer wants an £800 refund in March. Who signs it off?"
AGENT_RECURSION_LIMIT = 25

SEARCH_DIR, MODEL, PROVIDER

(PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before'),
 'gpt-oss-20b',
 'openai_compat')

## 3. Graph DB и recall (код репозитория, без LLM)

In [26]:
import sys

sys.path.insert(0, str(ROOT / "src"))

from build_graph import main as build_graph
from recall import recall

build_graph()
facts = recall(QUESTION)
memory_text = facts.as_text()
print(memory_text)

built graph.db: 13 entities, 13 relations, 10 aliases from 8 docs
memory: 8 facts recalled in 4 ms

Customer support SOP --[references]--> Refund approvals   (customer-support-sop.md)
Refund approvals --[approved_by]--> Ops Manager           (refund-policy.md)
Customer support SOP --[references]--> Support Lead       (customer-support-sop.md)
Onboarding process --[references]--> Ops Manager          (onboarding-process.md)
Ops Manager --[held_by]--> Sarah Chen                     (org-chart.md)
Sarah Chen --[delegates_to]--> Marcus Webb                (delegation-memo.md)
Incident response --[references]--> Support Lead          (incident-response.md)
Onboarding process --[references]--> Tooling inventory    (onboarding-process.md)

where:
  Customer support SOP: Tickets answered within one business day; refunds under £500 processed by agents; escalation after two replies
  Incident response: Incidents triaged by the Support Lead; severity-1 escalates to the Founder; write-up within 48

## 4. LangChain-модель

`make_chat_model()` — единственная точка смены бэкенда. Search и graph берут один и тот же объект.

In [27]:
from langchain_ollama import ChatOllama


def make_chat_model():
    if PROVIDER == "ollama":
        return ChatOllama(
            model=MODEL,
            temperature=0,
            base_url=OLLAMA_URL,
        )
    if PROVIDER == "openai_compat":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=MODEL,
            temperature=0,
            base_url=OPENAI_BASE,
            api_key=OPENAI_API_KEY,
        )
    raise ValueError(f"Unknown PROVIDER={PROVIDER!r}. Use 'ollama' or 'openai_compat'.")


llm = make_chat_model()
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x1131d4e90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x125218290>, root_client=<openai.OpenAI object at 0x1728bfb50>, root_async_client=<openai.AsyncOpenAI object at 0x173152210>, model_name='gpt-oss-20b', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://192.168.0.55:1234/v1', stream_chunk_timeout=120.0)

## 5. Инструменты search (Grep + Read), корень только `corpus-before/`

In [28]:
import re
from langchain.tools import tool

SEARCH_ROOT = SEARCH_DIR.resolve()


def _safe_md(rel: str) -> Path:
    path = (SEARCH_ROOT / rel).resolve()
    if not path.is_relative_to(SEARCH_ROOT) or path.suffix != ".md" or not path.is_file():
        raise ValueError(f"blocked path: {rel}")
    return path


@tool
def grep_notes(pattern: str) -> str:
    """Search markdown notes with a Python regex (case-insensitive). Returns matching lines."""
    rx = re.compile(pattern, re.I)
    hits = []
    for path in sorted(SEARCH_ROOT.glob("*.md")):
        for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if rx.search(line):
                hits.append(f"{path.name}:{i}:{line.strip()}")
    return "\n".join(hits[:80]) or "(no matches)"


@tool
def read_note(filename: str) -> str:
    """Read the full text of one markdown note. Pass only the filename, e.g. team.md."""
    return _safe_md(filename).read_text(encoding="utf-8")


TOOLS = [grep_notes, read_note]
list(SEARCH_ROOT.glob("*.md"))

[PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/ooo-march-sarah.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/incident-log-january.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/it-setup.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/newsletter-march-draft.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/expenses.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/holiday-rota-2026.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/supplier-notes.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/customer-ops-handbook.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpus-before/brand-voice.md'),
 PosixPath('/Users/semenoffalex/Agents/Cursor/graph-memory-starter/corpu

## 6. Search — агент LangChain (`create_agent`)

In [30]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, ToolMessage

SEARCH_SYSTEM = """You answer from company markdown notes using tools.
Use grep_notes to find candidates, then read_note for full files.
Do not guess names that you did not read. Be concise."""

search_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SEARCH_SYSTEM,
)

search_result = search_agent.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    {"recursion_limit": AGENT_RECURSION_LIMIT},
)


def last_text(messages) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage) and (m.content or "") and not m.tool_calls:
            c = m.content
            return c if isinstance(c, str) else str(c)
    return ""


def trace_tools(messages):
    names, reads = [], []
    for m in messages:
        for tc in getattr(m, "tool_calls", None) or []:
            name = tc["name"] if isinstance(tc, dict) else tc.name
            args = tc["args"] if isinstance(tc, dict) else tc.args
            names.append(name)
            if name == "read_note":
                reads.append(args.get("filename", ""))
    return names, sorted(set(r for r in reads if r))


search_answer = last_text(search_result["messages"])
search_tool_names, search_docs = trace_tools(search_result["messages"])
print(search_answer)
print("---")
print("tool calls:", search_tool_names)
print("docs read:", search_docs)

The **Operations Manager** must sign off an £800 refund before Finance processes the payment.
---
tool calls: ['grep_notes', 'read_note']
docs read: ['customer-ops-handbook.md']


## 7. Graph — тот же LLM, без инструментов, факты из recall

In [31]:
from langchain.messages import HumanMessage, SystemMessage

GRAPH_SYSTEM = (
    "Answer only from the memory block in the user message. "
    "Do not use tools. Do not invent facts. Be concise."
)
graph_prompt = f"{memory_text}\n\nQuestion: {QUESTION}"
graph_msg = llm.invoke(
    [SystemMessage(content=GRAPH_SYSTEM), HumanMessage(content=graph_prompt)]
)
graph_answer = graph_msg.content if isinstance(graph_msg.content, str) else str(graph_msg.content)
graph_contaminated = bool(getattr(graph_msg, "tool_calls", None))
print(graph_answer)
print("---")
print("tool_calls on graph cell:", getattr(graph_msg, "tool_calls", None))

Marcus Webb will sign the £800 refund off during Sarah Chen’s March leave.
---
tool_calls on graph cell: []


In [32]:
from langchain.messages import AIMessage


def usage_of(msg) -> dict:
    meta = getattr(msg, "usage_metadata", None) or {}
    resp = getattr(msg, "response_metadata", None) or {}
    tu = resp.get("token_usage") or resp.get("usage") or {}
    inp = meta.get("input_tokens") or tu.get("prompt_tokens") or tu.get("input_tokens")
    out = meta.get("output_tokens") or tu.get("completion_tokens") or tu.get("output_tokens")
    tot = meta.get("total_tokens") or tu.get("total_tokens")
    if tot is None and inp is not None and out is not None:
        tot = inp + out
    return {"input": inp or 0, "output": out or 0, "total": tot or 0}


def sum_usages(messages) -> dict:
    acc = {"input": 0, "output": 0, "total": 0, "calls": 0}
    for m in messages:
        if not isinstance(m, AIMessage):
            continue
        u = usage_of(m)
        if u["total"] == 0:
            continue
        acc["input"] += u["input"]
        acc["output"] += u["output"]
        acc["total"] += u["total"]
        acc["calls"] += 1
    return acc


search_usage = sum_usages(search_result["messages"])
graph_usage = usage_of(graph_msg)

print("search (сумма всех вызовов агента):", search_usage)
print("graph (один вызов):", graph_usage)
print(
    "delta total (search - graph):",
    search_usage["total"] - graph_usage["total"],
)

search (сумма всех вызовов агента): {'input': 2085, 'output': 230, 'total': 2315, 'calls': 3}
graph (один вызов): {'input': 468, 'output': 115, 'total': 583}
delta total (search - graph): 1732


## 8. Скоринг (протокол локального eval)

- **correct** — в ответе Marcus Webb (или однозначный Marcus) как тот, кто подписывает.
- hops: (1) £500 → Ops Manager, (2) роль у Sarah Chen, (3) март / Marcus.
- graph context read ≈ `len(memory_text) / 4`.

In [33]:
import pandas as pd
from langchain.messages import AIMessage

HOP1 = re.compile(r"(500|ops manager|operations manager)", re.I)
HOP2 = re.compile(r"sarah", re.I)
HOP3 = re.compile(r"marcus", re.I)


def hops(text: str) -> int:
    return sum(bool(rx.search(text or "")) for rx in (HOP1, HOP2, HOP3))


def usage_of(msg) -> dict:
    meta = getattr(msg, "usage_metadata", None) or {}
    resp = getattr(msg, "response_metadata", None) or {}
    tu = resp.get("token_usage") or resp.get("usage") or {}
    inp = meta.get("input_tokens") or tu.get("prompt_tokens") or tu.get("input_tokens")
    out = meta.get("output_tokens") or tu.get("completion_tokens") or tu.get("output_tokens")
    tot = meta.get("total_tokens") or tu.get("total_tokens")
    if tot is None and inp is not None and out is not None:
        tot = inp + out
    return {"input": inp or 0, "output": out or 0, "total": tot or 0}


def sum_usages(messages) -> dict:
    acc = {"input": 0, "output": 0, "total": 0, "calls": 0}
    for m in messages:
        if not isinstance(m, AIMessage):
            continue
        u = usage_of(m)
        if u["total"] == 0:
            continue
        acc["input"] += u["input"]
        acc["output"] += u["output"]
        acc["total"] += u["total"]
        acc["calls"] += 1
    return acc


def row(condition, answer, tool_names, docs, usage):
    n = hops(answer)
    return {
        "model": MODEL,
        "provider": PROVIDER,
        "condition": condition,
        "result": "correct" if HOP3.search(answer or "") else "wrong",
        "hops": f"{n} of 3",
        "tool_calls": len(tool_names),
        "docs_read": len(docs),
        "llm_calls": usage.get("calls", 1),
        "prompt_tokens": usage["input"],
        "completion_tokens": usage["output"],
        "total_tokens": usage["total"],
        "answer": (answer or "").strip().replace("\n", " ")[:240],
    }


if graph_contaminated:
    raise RuntimeError("Graph cell invoked tools — rerun; protocol requires 0 retrieval.")

search_usage = sum_usages(search_result["messages"])
graph_usage = usage_of(graph_msg)
graph_usage["calls"] = 1

table = pd.DataFrame(
    [
        row("search", search_answer, search_tool_names, search_docs, search_usage),
        row("graph", graph_answer, [], [], graph_usage),
    ]
)
display(table)

# table.to_csv("eval_local_results.csv", index=False)

,model,provider,condition,result,hops,tool_calls,docs_read,llm_calls,prompt_tokens,completion_tokens,total_tokens,answer
0,gpt-oss-20b,openai_compat,search,wrong,1 of 3,2,1,3,2085,230,2315,The **Operations Manager** must sign off an £8...
1,gpt-oss-20b,openai_compat,graph,correct,2 of 3,0,0,1,468,115,583,Marcus Webb will sign the £800 refund off duri...


In [ ]:
#table.to_csv("eval_local_results.csv", index=False)

In [34]:
from datetime import datetime
from pathlib import Path

OUT = Path(ROOT) / "notebooks" / "eval_local_results.csv" if "ROOT" in dir() else Path("eval_local_results.csv")

run = table.copy()
run.insert(0, "run_at", datetime.now().isoformat(timespec="seconds"))

if OUT.exists():
    prev = pd.read_csv(OUT)
    combined = pd.concat([prev, run], ignore_index=True)
else:
    combined = run

combined.to_csv(OUT, index=False)
display(combined)
print("wrote", OUT.resolve())

,model,provider,condition,result,hops,tool_calls,docs_read,llm_calls,prompt_tokens,completion_tokens,total_tokens,answer,run_at
0,hermes-3-llama-3.1-8b,openai_compat,search,wrong,0 of 3,2,0,3,1760,267,2027,Apologies for the confusion. Based on the limi...,NaN
1,hermes-3-llama-3.1-8b,openai_compat,graph,correct,3 of 3,0,0,1,414,41,455,"Sarah Chen, the Ops Manager, and Marcus Webb. ...",NaN
2,google/gemma-4-e2b,openai_compat,search,wrong,1 of 3,2,0,3,1199,901,2100,The refund must be signed off by the **Operati...,2026-09-04T11:28:12
3,google/gemma-4-e2b,openai_compat,graph,correct,1 of 3,0,0,1,446,358,804,Marcus Webb,2026-09-04T11:28:12
4,gpt-oss-20b,openai_compat,search,wrong,1 of 3,2,1,3,2085,230,2315,The **Operations Manager** must sign off an £8...,2026-09-04T11:43:41
5,gpt-oss-20b,openai_compat,graph,correct,2 of 3,0,0,1,468,115,583,Marcus Webb will sign the £800 refund off duri...,2026-09-04T11:43:41


wrote /Users/semenoffalex/Agents/Cursor/graph-memory-starter/eval_local_results.csv
